# Train on unscaled Figshare 141-point splits

This notebook trains the same eight-element FEFF pipeline as `train_figshare_141_splits.ipynb`: a scratch M3GNet encoder, a UniversalXAS head, and eight tuned heads. It uses Figshare column 4 directly, with no xsedge-derived or other intensity normalization. The exporter retains the established positive-value filter, 35 eV window, 0.25 eV grid, and material-level train/validation/test split.

Early stopping and checkpoint selection use validation data; test metrics are reported after training. Since these targets are deliberately unscaled, this notebook evaluates only the unscaled Figshare splits. Do not compare its intensity metrics with the prior calibrated Figshare run or Materials Cloud targets. The first run featurizes the new splits and can take a while. Start Jupyter from the repository root or `tutorial_omnixas`.


In [ ]:
from pathlib import Path
import json
import os
import random
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

cwd = Path.cwd().resolve()
if (cwd / 'tutorial_omnixas').is_dir():
    REPO_ROOT = cwd
elif cwd.name == 'tutorial_omnixas' and (cwd.parent / 'omnixas').is_dir():
    REPO_ROOT = cwd.parent
else:
    raise FileNotFoundError('Start Jupyter from the OmniXAS repository root or tutorial_omnixas directory.')
DATA_ROOT = Path(os.environ.get('OMNIXAS_DATA_ROOT', REPO_ROOT.parent / 'OmniXAS_data')).expanduser()
RAW_ROOT = DATA_ROOT / 'figshare'
NEW_ROOT = DATA_ROOT / 'figshare_141_splits_unscaled'
ELEMENTS = ['Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu']
SPLITS = ['train', 'val', 'test']
RUN_NAME = 'figshare_141_unscaled_seed42'
OUTPUT_ROOT = REPO_ROOT / 'output' / 'training' / 'figshare_141_unscaled_pipeline'
RUN_DIR = OUTPUT_ROOT / RUN_NAME
RESUME_RUN = False
OVERWRITE_RUN = False  # set True only when intentionally replacing this run
SEED = 42

required = [
    REPO_ROOT / 'models' / 'M3GNet-MP-2021.2.8-PES' / 'model.pt',
    REPO_ROOT / 'omnixas' / 'featurizer' / 'm3gnet_featurizer.py',
    RAW_ROOT / 'manifest.csv',
]
for path in required:
    if not path.exists():
        raise FileNotFoundError(f'Required input is missing: {path}')
split_files = [
    NEW_ROOT / folder / filename
    for folder, suffix in (('material_id_and_site', ''), ('ml_data', '_y'))
    for element in ELEMENTS
    for split in SPLITS
    for filename in [f'{element}_FEFF_{split}{suffix}.txt']
]
missing_splits = [path for path in split_files if not path.is_file()]
if missing_splits:
    existing_outputs = [path for path in NEW_ROOT.rglob('*') if path.is_file()] if NEW_ROOT.exists() else []
    if existing_outputs:
        raise FileNotFoundError(f'Unscaled Figshare splits are incomplete or stale ({len(missing_splits)} required files missing); move or remove {NEW_ROOT}, then rerun this cell.')
    exporter = REPO_ROOT / 'tutorial_omnixas' / 'export_figshare_141_splits_unscaled.py'
    print(f'Generating unscaled Figshare splits from {RAW_ROOT / "manifest.csv"}...')
    subprocess.run([sys.executable, str(exporter)], cwd=REPO_ROOT, check=True)
    missing_splits = [path for path in split_files if not path.is_file()]
    if missing_splits:
        raise FileNotFoundError(f'Split export did not create all required files; first missing file: {missing_splits[0]}')

report_path = NEW_ROOT / 'report.json'
manifest_path = NEW_ROOT / 'manifest.csv'
if not report_path.is_file() or not manifest_path.is_file():
    raise FileNotFoundError(f'Unscaled exporter report/manifest missing under {NEW_ROOT}')
report = json.loads(report_path.read_text(encoding='utf-8'))
if report.get('intensity_normalization') != 'none (factor 1.0; no xsedge-derived scale)':
    raise ValueError(f'Unexpected target normalization in {report_path}: {report.get("intensity_normalization")!r}')
row_manifest = pd.read_csv(manifest_path)
if not np.allclose(pd.to_numeric(row_manifest['target_scale_factor']), 1.0, rtol=0, atol=0):
    raise ValueError('Unscaled Figshare manifest contains a target scale other than 1.0')

if RESUME_RUN and OVERWRITE_RUN:
    raise ValueError('Choose either RESUME_RUN or OVERWRITE_RUN, not both.')
if RESUME_RUN:
    if not RUN_DIR.is_dir():
        raise FileNotFoundError(f'Cannot resume; run directory does not exist: {RUN_DIR}')
    if not (RUN_DIR / 'best_encoder.ckpt').is_file() and not (RUN_DIR / 'encoder_checkpoints' / 'last.ckpt').is_file():
        raise FileNotFoundError(f'Cannot resume; no saved encoder checkpoint in {RUN_DIR}. Set OVERWRITE_RUN=True to restart, or choose a new RUN_NAME.')
elif RUN_DIR.exists() and not OVERWRITE_RUN:
    raise FileExistsError(f'Run directory already exists; set RESUME_RUN=True to continue or OVERWRITE_RUN=True to replace it: {RUN_DIR}')

def read_ids(path):
    rows = [line.strip() for line in path.read_text(encoding='utf-8-sig').splitlines() if line.strip()]
    if not rows:
        raise ValueError(f'Empty ID split: {path}')
    return rows

def read_matrix(path, cols, label):
    if not path.is_file():
        raise FileNotFoundError(f'Missing {label}: {path}')
    array = np.loadtxt(path, dtype=np.float32, ndmin=2)
    if array.shape[1] != cols or not np.isfinite(array).all():
        raise ValueError(f'{label} must be finite with {cols} columns; got {array.shape}: {path}')
    if (array < 0).any():
        raise ValueError(f'{label} must be nonnegative: {path}')
    return array

# Verify targets, IDs, manifest ordering, material separation, and structure files.
new_data = {}
for element in ELEMENTS:
    material_sets = {}
    for split in SPLITS:
        key = (element, split)
        ids_path = NEW_ROOT / 'material_id_and_site' / f'{element}_FEFF_{split}.txt'
        y_path = NEW_ROOT / 'ml_data' / f'{element}_FEFF_{split}_y.txt'
        new_ids = read_ids(ids_path)
        new_y = read_matrix(y_path, 141, 'unscaled Figshare targets')
        if len(new_ids) != len(new_y):
            raise ValueError(f'ID/target row mismatch for {key}: {len(new_ids)} != {len(new_y)}')
        rows = row_manifest[(row_manifest['element'] == element) & (row_manifest['split'] == split)].sort_values('row_index')
        if rows['material_id_and_site'].tolist() != new_ids:
            raise ValueError(f'Manifest/ID ordering mismatch for {key}')
        if not np.allclose(new_y.max(axis=1), new_y.max(axis=1), rtol=0, atol=0):
            raise ValueError(f'Invalid target values for {key}')
        materials = set()
        for value in new_ids:
            try:
                material_id, site = value.rsplit('_', 1)
                site_number = int(site)
            except ValueError as exc:
                raise ValueError(f'Invalid split site ID {value!r} in {key}') from exc
            materials.add(material_id)
            poscar = RAW_ROOT / 'FEFF' / element / material_id / 'FEFF-XANES' / f'{site_number:03d}_{element}' / 'POSCAR'
            if not poscar.is_file():
                raise FileNotFoundError(f'No POSCAR for {value}: {poscar}')
        material_sets[split] = materials
        new_data[key] = {'ids': new_ids, 'y': new_y}
    for i, split in enumerate(SPLITS):
        for other in SPLITS[i + 1:]:
            overlap = material_sets[split] & material_sets[other]
            if overlap:
                raise ValueError(f'Material leakage for {element} between {split} and {other}: {sorted(overlap)[:5]}')
    print(f'{element}: train/val/test = {[len(new_data[(element, s)]["ids"]) for s in SPLITS]}')

if RUN_DIR.exists() and not RESUME_RUN and not OVERWRITE_RUN:
    raise FileExistsError(f'Run directory was created while preparing inputs; set RESUME_RUN=True to continue or OVERWRITE_RUN=True to replace it: {RUN_DIR}')


## Train the unscaled Figshare encoder and heads

This runs the same scratch-encoder, balanced UniversalXAS, and tuned-head training pipeline as the calibrated Figshare notebook. The only data-target change is that the exporter passes Figshare column 4 through without multiplying by a fitted FEFF scale.


In [ ]:
import importlib
import runpy
from omnixas.data import feff_graph
importlib.reload(feff_graph)


SCRIPT = REPO_ROOT / 'tutorial_omnixas' / 'train_m3gnet_xas_pipeline.py'
OUTPUT_ROOT = REPO_ROOT / 'output' / 'training' / 'figshare_141_unscaled_pipeline'
RUN_NAME = 'figshare_141_unscaled_seed42'
RUN_DIR = OUTPUT_ROOT / RUN_NAME

# Keep graph batches small enough for a 24 GB GPU. The encoder batch is 8 * 16 = 128 sites.
script_args = [
    str(SCRIPT),
    '--dataset-root', str(NEW_ROOT),
    '--raw-root', str(DATA_ROOT / 'figshare'),
    '--output-root', str(OUTPUT_ROOT),
    '--run-name', RUN_NAME,
    '--precision', 'bf16-mixed',
    '--encoder-rows-per-element', '16',
    '--encoder-eval-batch-size', '64',
    '--num-workers', '16',
    '--batch-size', '4096',
]
if RESUME_RUN:
    script_args.append('--resume')
elif OVERWRITE_RUN:
    script_args.append('--overwrite')
previous_argv = sys.argv
sys.argv = script_args
try:
    runpy.run_path(str(SCRIPT), run_name='__main__')
finally:
    sys.argv = previous_argv


## Evaluate on unscaled Figshare splits

Report validation and test metrics for the UniversalXAS and tuned heads. The training targets are unscaled Figshare intensities, so scores use the matching unscaled Figshare train mean as the eta baseline.


In [ ]:
from omnixas.model.m3gnet_xas import XASSpectralHead

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for element in ELEMENTS:
    task = f'{element}_FEFF'
    for split in SPLITS:
        data = new_data[(element, split)]
        data['X'] = np.atleast_2d(np.loadtxt(RUN_DIR / 'features' / f'{task}_{split}_X.txt', dtype=np.float32))
        if data['X'].shape != (len(data['ids']), 64):
            raise ValueError(f'Invalid trained Figshare features for {task} {split}: {data["X"].shape}')

def load_head(path):
    head = XASSpectralHead().to(device)
    head.load_state_dict(torch.load(path, map_location='cpu', weights_only=False)['state_dict'], strict=True)
    return head.eval()

heads = {
    'UniversalXAS': load_head(RUN_DIR / 'heads' / 'universalXAS' / 'best.pt'),
    'Tuned-UniversalXAS': {element: load_head(RUN_DIR / 'heads' / 'tunedUniversalXAS' / f'{element}_FEFF' / 'best.pt') for element in ELEMENTS},
}

def predict(head, X):
    output = []
    with torch.inference_mode():
        for start in range(0, len(X), 4096):
            output.append(head(torch.as_tensor(X[start:start + 4096], dtype=torch.float32, device=device)).cpu().numpy())
    return np.concatenate(output, axis=0)

def score(element, variant, split_name, y_true, y_pred, train_mean):
    errors = np.mean((y_true - y_pred) ** 2, axis=1)
    baseline = np.mean((y_true - train_mean) ** 2, axis=1)
    return {
        'element': element, 'variant': variant, 'evaluation_split': split_name, 'n_spectra': len(y_true),
        'aggregate_mse': float(np.mean((y_true - y_pred) ** 2)), 'median_spectrum_mse': float(np.median(errors)),
        'baseline_median_mse': float(np.median(baseline)), 'eta': float(np.median(baseline) / max(np.median(errors), 1e-12)),
    }, errors

records, predictions = [], {}
for element in ELEMENTS:
    train = new_data[(element, 'train')]
    train_mean = train['y'].mean(axis=0, keepdims=True)
    for variant, head_source in heads.items():
        head = head_source if variant == 'UniversalXAS' else head_source[element]
        for split_name, data in [('val', new_data[(element, 'val')]), ('test', new_data[(element, 'test')])]:
            pred = predict(head, data['X'])
            row, errors = score(element, variant, split_name, data['y'], pred, train_mean)
            records.append(row)
            if variant == 'Tuned-UniversalXAS' and split_name == 'test':
                predictions[element] = {'y': data['y'], 'pred': pred, 'errors': errors}

results = pd.DataFrame(records).sort_values(['element', 'variant', 'evaluation_split'])
results.to_csv(RUN_DIR / 'evaluation_unscaled_figshare.csv', index=False)
display(results.round({'aggregate_mse': 6, 'median_spectrum_mse': 6, 'baseline_median_mse': 6, 'eta': 3}))


## Test spectrum percentiles

Each panel shows a test target and tuned-head prediction for a spectrum near the indicated percentile of per-spectrum MSE. These plots use the unscaled Figshare target values.


In [ ]:
percentiles = [10, 20, 30, 40, 50, 60, 70, 80, 90]
energy = np.arange(141) * 0.25
for element in ELEMENTS:
    item = predictions[element]
    order = np.argsort(item['errors'])
    fig, axes = plt.subplots(3, 3, figsize=(12, 8), sharex=True, sharey=True)
    for ax, percentile in zip(axes.flat, percentiles):
        idx = order[int(round((percentile / 100) * (len(order) - 1)))]
        ax.plot(energy, item['y'][idx], label='target', linewidth=1.5)
        ax.plot(energy, item['pred'][idx], label='prediction', linewidth=1.2)
        ax.set_title(f'{percentile}th percentile (MSE={item["errors"][idx]:.3g})')
        ax.grid(alpha=0.2)
    fig.suptitle(f'{element} — unscaled Figshare test')
    fig.supxlabel('Energy above grid start (eV)')
    fig.supylabel('Unscaled intensity')
    axes.flat[0].legend(frameon=False)
    fig.tight_layout()
    fig.savefig(RUN_DIR / f'{element}_unscaled_test_percentiles.png', dpi=140, bbox_inches='tight')
    plt.show()
    plt.close(fig)
print(f'Unscaled Figshare results and plots saved in {RUN_DIR}')
